# Phase 6 - SageMaker Pipeline

This notebook builds the preprocessing → evaluation → quality gate pipeline. It does not create AWS resources until the upsert cell is run.

In [ ]:
import boto3
from pathlib import Path
from pipelines.sagemaker_pipeline import build_pipeline

REGION = 'ap-southeast-2'
ACCOUNT_ID = boto3.client('sts', region_name=REGION).get_caller_identity()['Account']
BUCKET = f'sagemaker-ml-lab-{ACCOUNT_ID}-{REGION}'
PIPELINE_NAME = 'customer-churn-phase6'
INPUT_S3_URI = f's3://{BUCKET}/datasets/customer_churn.example.csv'
ROLE = boto3.client('iam').get_role(RoleName='AmazonSageMakerAdminIAMExecutionRole')['Role']['Arn']
print(REGION, BUCKET, PIPELINE_NAME, ROLE)

In [ ]:
pipeline = build_pipeline(role=ROLE, bucket=BUCKET, region=REGION, pipeline_name=PIPELINE_NAME, default_input=INPUT_S3_URI)
definition = pipeline.definition()
print(f'Pipeline definition generated: {len(definition)} bytes')

## Explicit AWS actions

Run the next cell only after confirming the input CSV exists in S3. It creates/updates the pipeline definition.

In [ ]:
pipeline.upsert(role_arn=ROLE)
print('Pipeline upserted')

In [ ]:
execution = pipeline.start()
print('Execution ARN:', execution.arn)